# 📸 Image Captioning Part A: Vision Transformers & Cross-Attention

## What is Image Captioning?

Image captioning is the task of generating a natural language description of an image. It bridges **computer vision** and **natural language processing**.

### Encoder-Decoder Architecture

```
  [Image]
     |
  [Encoder]  <-- extracts visual features (patches → tokens)
     |
  [Cross-Attention]  <-- text queries attend to image features
     |
  [Decoder]  <-- generates caption token by token
     |
  "A dog running in a park"
```

### CNN vs ViT Encoder

| Feature | CNN | ViT (Vision Transformer) |
|---|---|---|
| Input representation | Sliding conv filters | Fixed-size patches |
| Receptive field | Local → global (deep layers) | Global from layer 1 |
| Position info | Implicit via structure | Explicit positional encoding |
| Inductive bias | Strong (translation equivariance) | Weak (needs more data) |
| Output | Feature maps (H×W×C) | Sequence of patch tokens |
| Scales with data | Moderate | Excellent |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Create a dummy 32x32 RGB image
np.random.seed(42)
image = np.random.randint(0, 255, (32, 32, 3), dtype=np.uint8)

# Patchify: split into 4x4 patches (patch_size=8 → 4x4=16 patches)
patch_size = 8
H, W, C = image.shape
num_patches_h, num_patches_w = H // patch_size, W // patch_size

# Reshape into patches and flatten each patch
patches = image.reshape(num_patches_h, patch_size, num_patches_w, patch_size, C)
patches = patches.transpose(0, 2, 1, 3, 4)  # (n_h, n_w, patch_size, patch_size, C)
patches_flat = patches.reshape(-1, patch_size * patch_size * C)  # (16, 192)

print(f"Original image shape: {image.shape}")
print(f"Number of patches:    {patches_flat.shape[0]}  ({num_patches_h}x{num_patches_w} grid)")
print(f"Each patch flattened: {patches_flat.shape[1]}  ({patch_size}x{patch_size}x{C})")

# Plot the patch grid
fig, axes = plt.subplots(num_patches_h, num_patches_w, figsize=(5, 5))
fig.suptitle(f"32×32 image split into {num_patches_h*num_patches_w} patches ({patch_size}×{patch_size} each)", fontsize=11)
for i in range(num_patches_h):
    for j in range(num_patches_w):
        axes[i, j].imshow(patches[i, j])
        axes[i, j].axis('off')
        axes[i, j].set_title(f"{i*num_patches_w+j}", fontsize=7)
plt.tight_layout()
plt.show()

## ViT Positional Encoding

Unlike CNNs, ViTs treat patches as a **flat sequence** — the model has no built-in sense of spatial layout. Without positional encoding, patch 0 (top-left) looks identical to patch 15 (bottom-right) to the transformer.

### 1D vs 2D Positional Encoding

**1D (standard ViT):** Assign a single index 0…N-1 to each patch, then learn or compute an embedding per index. Simple, works well in practice.

**2D:** Assign separate row and column encodings, then combine them. Preserves the grid structure more explicitly:

```
PE(row, col, 2i)   = sin(row / 10000^(2i/d))  +  sin(col / 10000^(2i/d))
PE(row, col, 2i+1) = cos(row / 10000^(2i/d))  +  cos(col / 10000^(2i/d))
```

**Why images need it:** A sentence "cat sat" vs "sat cat" changes meaning, just like a patch grid where spatial layout is semantically critical (sky is always top, ground always bottom).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def positional_encoding_2d(n_rows, n_cols, d_model):
    """2D sinusoidal positional encoding for image patches."""
    pe = np.zeros((n_rows, n_cols, d_model))
    for i in range(d_model // 2):
        denom = 10000 ** (2 * i / d_model)
        rows = np.arange(n_rows)[:, None]   # (n_rows, 1)
        cols = np.arange(n_cols)[None, :]   # (1, n_cols)
        pe[:, :, 2*i]   = np.sin(rows / denom) + np.sin(cols / denom)
        pe[:, :, 2*i+1] = np.cos(rows / denom) + np.cos(cols / denom)
    return pe.reshape(n_rows * n_cols, d_model)  # flatten to (N_patches, d_model)

pe = positional_encoding_2d(n_rows=4, n_cols=4, d_model=32)
print(f"Positional encoding shape: {pe.shape}  (16 patches, 32-dim embedding)")

plt.figure(figsize=(8, 2.5))
plt.imshow(pe, aspect='auto', cmap='RdBu')
plt.colorbar()
plt.xlabel("Embedding dimension")
plt.ylabel("Patch index (row-major)")
plt.title("2D Positional Encoding — each patch gets a unique signature")
plt.tight_layout()
plt.show()

## Cross-Attention: The Decoder Asks, the Encoder Answers

In cross-attention, **queries come from the decoder** (the text being generated) and **keys/values come from the encoder** (the image patches). The decoder is effectively asking: *"Which image regions are most relevant to the word I'm about to generate?"*

### Formula

$$\text{CrossAttention}(Q_{\text{text}},\ K_{\text{image}},\ V_{\text{image}}) = \text{softmax}\!\left(\frac{Q_{\text{text}}\ K_{\text{image}}^{T}}{\sqrt{d_k}}\right) V_{\text{image}}$$

- **Q** (queries) — from the text decoder tokens  
- **K** (keys) — from the image patch encoder  
- **V** (values) — from the image patch encoder  
- The attention weights show **which patches each word attends to**

For example, when generating the word *"dog"*, the attention weights should be high over the patch containing the dog.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
d_k, n_patches, n_text_tokens = 16, 16, 4
text_tokens = ["<start>", "a", "dog", "runs"]

# Random Q (text decoder), K and V (image encoder)
Q = np.random.randn(n_text_tokens, d_k)   # (4 words, 16-dim)
K = np.random.randn(n_patches, d_k)       # (16 patches, 16-dim)
V = np.random.randn(n_patches, d_k)       # (16 patches, 16-dim)

# Cross-attention: scores and weights
scores = Q @ K.T / np.sqrt(d_k)           # (4, 16)
scores -= scores.max(axis=1, keepdims=True)  # numerical stability
attn_weights = np.exp(scores) / np.exp(scores).sum(axis=1, keepdims=True)
output = attn_weights @ V                  # (4, 16)

print(f"Q shape: {Q.shape}  |  K shape: {K.shape}  |  Attention weights: {attn_weights.shape}")
print(f"Output shape: {output.shape}  (each text token now encodes relevant image info)")

# Plot 4x4 attention heatmap (each text token attending to 16 image patches)
fig, ax = plt.subplots(figsize=(7, 3))
im = ax.imshow(attn_weights, cmap='viridis', aspect='auto')
ax.set_yticks(range(n_text_tokens)); ax.set_yticklabels(text_tokens)
ax.set_xlabel("Image patch index (0–15)")
ax.set_title("Cross-Attention Weights: which patches each word attends to")
plt.colorbar(im, ax=ax, label="Attention weight")
plt.tight_layout()
plt.show()

## Quiz

Test your understanding of ViT and cross-attention for image captioning.

---

**Q1. Why does ViT require explicit positional encoding, but a standard CNN does not?**

<details>
<summary>Show answer</summary>

CNNs apply filters in a spatially structured way — the filter's position during the convolution operation implicitly encodes location. Transformers, by contrast, treat all patches as an unordered set (like a bag of words) and apply attention globally. Without adding positional information explicitly, patch 0 and patch 15 would be indistinguishable to the model.

</details>

---

**Q2. In cross-attention, why do the Queries come from the text decoder and not from the image encoder?**

<details>
<summary>Show answer</summary>

The decoder is trying to generate the next word. To do so intelligently, it needs to "ask" the image: *which visual regions are relevant right now?* The query represents the current context (what the decoder needs), while the keys/values represent what the image can offer. Swapping them would mean the image is querying the text, which doesn't serve the generation goal.

</details>

---

**Q3. If a 224×224 image is split into 16×16 patches, how many patch tokens does the ViT encoder receive?**

<details>
<summary>Show answer</summary>

(224 / 16) × (224 / 16) = 14 × 14 = **196 patch tokens**. Each token is a flattened 16×16×3 = 768-dimensional vector (before the linear projection).

</details>